# MathProver: Find or certify proofs for theorems

This notebook demonstrates the complete first version of MathProver. A language model may suggest a proof, but **Lean 4 is the verifier**: a result is certified only when Lean accepts the generated formal proof.

Run the cells from top to bottom. Before starting, activate the `mathprover` Python environment, install the project with `pip install -e .`, and select that environment as this notebook's kernel. The project also needs a working `lean_project/` directory with Mathlib installed.

There are two APIs:

- `certify(source)` checks Lean code that *you* supply.
- `prove(theorem_statement)` asks OpenAI for a proof and has Lean check it.

For `certify`, include `import Mathlib` and a complete proof. For `prove`, provide only a Lean theorem declaration—do not include `:= by`, a proof, or `import Mathlib`.

## 1. Certify a proof we supply

The next cell passes a complete Lean file to the local Lean compiler. `simp` proves that adding zero to a natural number changes nothing. A `True` result means Lean checked every part of the theorem and proof.

A successful Lean invocation commonly produces no diagnostic text; that is expected.

In [2]:
from mathprover import certify

source = """
import Mathlib

theorem add_zero_test (n : Nat) : n + 0 = n := by
  simp
"""

result = certify(source)

print("Certified:", result.certified)
print(result.stderr)

Certified: True



## 2. Confirm that Lean rejects a false claim

Certification is meaningful only if invalid proofs fail. The next cell attempts to prove `1 = 2` using `rfl`. Lean rejects the source, so `Certified` must be `False`; the error message is Lean's explanation.

This is why model output alone is never treated as a mathematical proof.

In [3]:
bad_source = """
import Mathlib

theorem false_claim : 1 = 2 := by
  rfl
"""

result = certify(bad_source)

print("Certified:", result.certified)
print(result.stderr)

Certified: False



## 3. Generate a proof, then certify it

The next cell supplies only a Lean theorem statement. MathProver loads `OPENAI_API_KEY` from the top-level `.env` file, requests a Lean proof beginning with `by`, adds the Mathlib import, and calls Lean locally. If Lean rejects a candidate, its diagnostic is supplied to the model on the next attempt.

The default is at most three attempts. Model calls can take noticeably longer than Lean checking. For a quick experiment, use `max_attempts=1`.

The input is formal Lean, not natural language. A future formalization step can translate an English claim into a theorem statement, but it should show the formalized statement for human review before proving it.

In [4]:
from mathprover import prove

result = prove("""theorem alg_formula_2_test (a b : Nat) : (a + b)*(a + b) = a*a + 2*a*b + b*b""",
              max_attempts = 2,
              verbose = 1)

print("Certified:", result.certified)
print("Attempts:", result.attempts)
print("Proof:")
print(result.proof)
print(result.lean_result.stderr)

[tactics] Trying 16 standard tactics in one compile...
[tactics] Closed by `ring`.
Certified: True
Attempts: 0
Proof:
ring



In [5]:
from mathprover import prove

result = prove("""theorem alg_formula_3_test (a b : Nat) : (a + b)*(a - b) = a*a - b*b""",
              max_attempts = 2,
              verbose = 1)

print("Certified:", result.certified)
print("Attempts:", result.attempts)
print("Proof:")
print(result.proof)
print(result.lean_result.stderr)

[tactics] Trying 16 standard tactics in one compile...
[tactics] Closed by `exact?`.
Certified: True
Attempts: 0
Proof:
exact?



## Example-1: Prove that $n^3 + 2n$ is divisible by $3$ for any natural number $n$

In [7]:
from mathprover import prove

result = prove("""theorem cube_add_two_mul (n : Nat) : 3 ∣ n^3 + 2*n""",
              max_attempts = 3,
              samples_per_attempt = 4,
              verbose = 1)

print("Certified:", result.certified)
print("Attempts:", result.attempts)
print("Proof:")
print(result.proof)
print(result.lean_result.stderr)

[tactics] Trying 16 standard tactics in one compile...
[tactics] None closed the goal.
[grounding] 10 of 11 proposed names exist.
[attempt 1/3] Requesting 4 candidate(s) from gpt-5...
[attempt 1/3] Checking candidate 1/4 with Lean...
[attempt 1/3] Lean rejected candidate 1.
[attempt 1/3] Checking candidate 2/4 with Lean...
[attempt 1/3] Lean certified the proof.
Certified: True
Attempts: 1
Proof:
classical
set q : Nat := n / 3
set r : Nat := n % 3
have hn : n = 3 * q + r := by
  have : n = n % 3 + 3 * (n / 3) := by simpa using (Nat.mod_add_div n 3).symm
  simpa [q, r, Nat.mul_comm, Nat.add_comm, Nat.mul_left_comm, Nat.add_left_comm, Nat.add_assoc] using this
have hr0 : 0 ≤ r := Nat.zero_le _
have hrlt3 : r < 3 := by simpa [r] using (Nat.mod_lt n (by decide : 0 < 3))
have hrle2 : r ≤ 2 := (Nat.lt_succ_iff.mp (by simpa [Nat.succ_eq_add_one] using hrlt3))
interval_cases r using hr0, hrle2
· subst r
  have hn0 : n = 3 * q := by simpa [Nat.add_comm] using hn
  refine ⟨9 * q ^ 3 + 2 * q, ?_⟩

## Example-2: Let $p$ be an odd prime and let $a$ be a nonzero residue modulo $p$. Then $a^{(p-1)/2} \equiv \pm 1 \pmod{p}$

In [1]:
from mathprover import prove

result = prove("""theorem euler_dichotomy (p : ℕ) [Fact p.Prime] (hp : p ≠ 2) (a : ZMod p) (ha : a ≠ 0) : a^((p-1)/2) = 1 ∨ a^((p-1)/2) = -1""",
               model="gpt-6-astra",
               reasoning_effort="xhigh",
               max_attempts = 3,
               samples_per_attempt = 4,
               verbose = 1)

print("Certified:", result.certified)
print("Attempts:", result.attempts)
print("Proof:")
print(result.proof)
print(result.lean_result.stderr)

[tactics] Trying 16 standard tactics in one compile...
[tactics] None closed the goal.
[grounding] 7 of 7 proposed names exist.
[attempt 1/3] Requesting 4 candidate(s) from gpt-6-astra...
[attempt 1/3] Checking candidate 1/4 with Lean...
[attempt 1/3] Lean certified the proof.
Certified: True
Attempts: 1
Proof:
apply sq_eq_one_iff.mp
have hp_prime : p.Prime := Fact.out
have hdiv : 2 ∣ p - 1 := by
  obtain ⟨k, hk⟩ := hp_prime.odd_of_ne_two hp
  omega
rw [← pow_mul, Nat.div_mul_cancel hdiv]
apply ZMod.pow_card_sub_one_eq_one
exact ha



## What to try next

Change the theorem statement to another small result from natural-number algebra. Keep the theorem declaration syntactically complete, but omit `:= by`. Record whether it certifies, how many attempts it used, and Lean's diagnostic if it fails.